<a href="https://colab.research.google.com/github/AyandaMnyango/AyandaMnyango/blob/main/Real_time_Sentiment_Analysis_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import tweepy
import pandas as pd
from datetime import datetime
import os
from dotenv import load_dotenv

load_dotenv()

class TwitterScraper:
    def __init__(self):
        self.client = tweepy.Client(
            bearer_token=os.getenv('TWITTER_BEARER_TOKEN'),
            wait_on_rate_limit=True
        )

    def search_tweets(self, query, max_results=100, days_back=7):
        """Search recent tweets by query"""
        tweets = tweepy.Paginator(
            self.client.search_recent_tweets,
            query=query,
            tweet_fields=['created_at', 'public_metrics', 'lang'],
            max_results=min(max_results, 100)
        ).flatten(limit=max_results)

        data = []
        for tweet in tweets:
            data.append({
                'id': tweet.id,
                'text': tweet.text,
                'created_at': tweet.created_at,
                'retweets': tweet.public_metrics['retweet_count'],
                'likes': tweet.public_metrics['like_count'],
                'lang': tweet.lang
            })

        return pd.DataFrame(data)

    def stream_tweets(self, keywords, callback):
        """Real-time streaming (requires Elevated access)"""
        streaming_client = TweetStreamingClient(callback)
        streaming_client.add_rules(tweepy.StreamRule(" OR ".join(keywords)))
        streaming_client.filter()

In [11]:
import praw
import pandas as pd
from datetime import datetime
import os

class RedditScraper:
    def __init__(self):
        self.reddit = praw.Reddit(
            client_id=os.getenv('REDDIT_CLIENT_ID'),
            client_secret=os.getenv('REDDIT_CLIENT_SECRET'),
            user_agent=os.getenv('REDDIT_USER_AGENT')
        )

    def search_subreddit(self, subreddit_name, query=None, limit=100, sort='hot'):
        """Fetch posts from subreddit"""
        subreddit = self.reddit.subreddit(subreddit_name)

        if sort == 'hot':
            posts = subreddit.hot(limit=limit)
        elif sort == 'new':
            posts = subreddit.new(limit=limit)
        else:
            posts = subreddit.search(query, limit=limit) if query else subreddit.hot(limit=limit)

        data = []
        for post in posts:
            data.append({
                'id': post.id,
                'title': post.title,
                'text': post.selftext,
                'subreddit': subreddit_name,
                'score': post.score,
                'num_comments': post.num_comments,
                'created_utc': datetime.fromtimestamp(post.created_utc),
                'url': post.url
            })

        return pd.DataFrame(data)

    def get_comments(self, post_id, limit=50):
        """Fetch comments from a specific post"""
        submission = self.reddit.submission(id=post_id)
        submission.comments.replace_more(limit=0)

        comments = []
        for comment in submission.comments.list()[:limit]:
            comments.append({
                'id': comment.id,
                'text': comment.body,
                'score': comment.score,
                'created_utc': datetime.fromtimestamp(comment.created_utc)
            })

        return pd.DataFrame(comments)

In [6]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import re
import emoji
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt')

# Cell 2: Load Data
df_twitter = pd.read_csv('../data/raw/twitter_data.csv')
df_reddit = pd.read_csv('../data/raw/reddit_data.csv')

# Cell 3: Text Cleaning Functions
class TextCleaner:
    def __init__(self):
        self.stop_words = set(stopwords.words('english'))

    def clean_text(self, text):
        """Full cleaning pipeline"""
        if pd.isna(text):
            return ""

        # Lowercase
        text = text.lower()

        # Remove URLs
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

        # Remove user mentions and hashtags (keep hashtag text)
        text = re.sub(r'@\w+', '', text)
        text = re.sub(r'#(\w+)', r'\1', text)

        # Remove emojis (or convert to text)
        text = emoji.demojize(text)

        # Remove special characters, keep alphanumeric and spaces
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

        # Remove extra whitespace
        text = ' '.join(text.split())

        return text

    def tokenize_and_remove_stopwords(self, text):
        """Tokenize and remove stopwords"""
        tokens = word_tokenize(text)
        tokens = [t for t in tokens if t not in self.stop_words and len(t) > 2]
        return tokens

# Apply cleaning
cleaner = TextCleaner()
df_twitter['cleaned_text'] = df_twitter['text'].apply(cleaner.clean_text)
df_reddit['cleaned_text'] = (df_reddit['title'].fillna('') + ' ' +
                              df_reddit['text'].fillna('')).apply(cleaner.clean_text)

# Cell 4: EDA Visualizations
# Distribution of text lengths
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
df_twitter['text_length'] = df_twitter['cleaned_text'].str.len()
sns.histplot(df_twitter['text_length'], bins=50, kde=True)
plt.title('Twitter Text Length Distribution')

plt.subplot(1, 2, 2)
df_reddit['text_length'] = df_reddit['cleaned_text'].str.len()
sns.histplot(df_reddit['text_length'], bins=50, kde=True)
plt.title('Reddit Text Length Distribution')
plt.tight_layout()
plt.show()

# Word Cloud
all_text = ' '.join(df_twitter['cleaned_text'].tolist())
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_text)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Most Common Words')
plt.show()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/twitter_data.csv'

In [8]:
pip install praw


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 3.3 MB/s eta 0:00:00
